# 01 — Setup and Data Acquisition

Este notebook valida a aquisição dos dados FAERS usados no projeto.

Os ficheiros foram carregados para o volume Databricks:

`/Volumes/main/default/faers_data/`

Esta pasta será usada como fonte de dados brutos. A partir deste ponto, os ficheiros originais não serão modificados.

In [0]:
# criação de caminho base e verificação que os dados foram carregados corretamente para o caminho definido 

faers_base_path = "/Volumes/main/default/faers_data/"

display(dbutils.fs.ls(faers_base_path))

## 1.1 Validação da estrutura da fonte

Após a aquisição, o diretório raw contém as seguintes pastas FAERS: `DEMO`, `DRUG`, `REAC`, `OUTC` e `THER`.

Estas pastas correspondem às principais entidades do dataset FAERS. 

In [0]:
# validação de que todos os dados dos trimestres necessários estão presentes nas pastas correspondentes
selected_tables = ["DEMO", "DRUG", "REAC", "OUTC"]
for table in selected_tables:
    print(f"\n--- {table} ---")
    display(dbutils.fs.ls(f"{faers_base_path}{table}/"))

## 1.2 Seleção das tabelas FAERS

Para o projeto foram selecionadas inicialmente quatro tabelas principais do dataset FAERS: 

- `DEMO`: contém informação geral do relatório/caso e dados demográficos do paciente;
- `DRUG`: contém os medicamentos associados a cada relatório;
- `REAC`: contém as reações adversas reportadas.
- `OUTC`: contém desfechos clínicos associados aos casos reportados.

Estas quatro tabelas permitem relacionar casos, medicamentos e eventos adversos através de identificadores comuns como `primaryid` e `caseid`.

Nesta fase foi realizada uma leitura exploratória das tabelas selecionadas (`DEMO`, `DRUG`,`REAC` e `OUTC`) apenas para confirmar que os ficheiros raw são interpretados corretamente com o delimitador `$`.

Como o objetivo desta etapa é apenas validar a estrutura e visualizar amostras, os tipos de dados ainda não foram convertidos. O carregamento definitivo para a camada Bronze será feito posteriormente com schemas explícitos.

## Conclusão do Setup Inicial

A etapa de setup e aquisição confirmou que os dados FAERS necessários para o projeto estão disponíveis em `/Volumes/main/default/faers_data/`.

Foram selecionadas quatro tabelas principais para análise: `DEMO`, `DRUG`, `REAC` e `OUTC`. 
Estas tabelas permitem relacionar relatórios, medicamentos, reações adversas e desfechos clínicos.

A leitura exploratória confirmou que os ficheiros raw são lidos corretamente com cabeçalho e delimitador `$`. Nesta fase, não foram aplicadas transformações aos dados nem definidos schemas para cada tabela. 

# 02 — Bronze Loading


Este notebook cria a camada Bronze do projeto FAERS.

De acordo com a arquitetura medalhão, a camada Bronze corresponde à conversão dos ficheiros raw para tabelas Delta, mantendo os dados o mais próximo possível da origem. Os dados foram lidos a partir de `/Volumes/main/default/faers_data/` e são guardados em `/Volumes/main/default/faers_data/delta/bronze/`.

Nesta etapa são aplicados schemas explícitos e adicionadas colunas de metadata técnica:

- `source_file`: identifica o ficheiro raw de origem de cada registo;
- `load_timestamp`: regista o momento em que os dados foram carregados para Bronze.

Não são aplicadas limpezas, deduplicações ou normalizações nesta camada. Essas operações serão realizadas posteriormente na camada Silver.




###*Estratégia de schema*

Na camada Bronze, todas as colunas vão ser  carregadas como `string` de forma temporária.

Esta decisão preserva os valores originais dos ficheiros raw e evita perdas ou conversões incorretas na primeira etapa do pipeline especialmente em campos como datas, idades, pesos, identificadores e códigos clínicos. Apesar de os tipos de dados serem preservados como texto, o schema continua a ser definido de forma explícita.

*A camada Bronze tem como objetivo materializar os dados de forma próxima do estado puro em formato Delta, com schema explícito e acrescentar dias colunas de metadata.* 

Vamos definir uma função auxiliar de uso único apenas para passar o schema temporário a todas as tabelas para facilitar leitura de código. De seguida foram gravadas todas as tabelas, com a adição de duas colunas para registar metadados, em formato delta de forma automática com recurso a um for_loop

Por fim é feita validação de que todas as tabelas foram guardadas no formato e schema correto e que as colunas criadas foram adicionadas. 



In [0]:
# imports necessários
from pyspark.sql.functions import input_file_name, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType

In [0]:
# criação de função auxiliar para definir o schema de uma tabela manualmente

from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import col, current_timestamp


def create_string_schema(columns):
    return StructType([
        StructField(column, StringType(), True)
        for column in columns
    ])


In [0]:
# criação de listas de colunas para cada uma das tabelas 

demo_columns = [
    "primaryid", "caseid", "caseversion", "i_f_code",
    "event_dt", "mfr_dt", "init_fda_dt", "fda_dt",
    "rept_cod", "auth_num", "mfr_num", "mfr_sndr",
    "lit_ref", "age", "age_cod", "age_grp", "sex", "e_sub",
    "wt", "wt_cod", "rept_dt", "to_mfr",
    "occp_cod", "reporter_country", "occr_country"
]

drug_columns = [
    "primaryid", "caseid", "drug_seq", "role_cod",
    "drugname", "prod_ai", "val_vbm", "route",
    "dose_vbm", "cum_dose_chr", "cum_dose_unit",
    "dechal", "rechal", "lot_num", "exp_dt",
    "nda_num", "dose_amt", "dose_unit",
    "dose_form", "dose_freq"
]

reac_columns = [
    "primaryid", "caseid", "pt", "drug_rec_act"
]

outc_columns = [
    "primaryid", "caseid", "outc_cod"
]


In [0]:
# passagem do schema para cada uma das tabelas

schemas = {
    "DEMO": create_string_schema(demo_columns),
    "DRUG": create_string_schema(drug_columns),
    "REAC": create_string_schema(reac_columns),
    "OUTC": create_string_schema(outc_columns)
}

In [0]:
# carregamento do schema para todas as tabelas e adição de colunas com metadata(source_file, load_timestamp)
bronze_dfs = {}

for table_name in selected_tables:
    input_path = f"{faers_base_path}{table_name}/"
    schema = schemas[table_name]
    
    df = (
        spark.read
        .option("header", "true")
        .option("delimiter", "$")
        .schema(schema)
        .csv(input_path)
        .withColumn("source_file", col("_metadata.file_path")) # Adição de coluna com nome do ficheiro
        .withColumn("load_timestamp", current_timestamp()) # Adição de coluna com timestamp
    )
    
    bronze_dfs[table_name] = df
    
    print(f"\nTabela raw carregada para DataFrame Bronze: {table_name}")
 

In [0]:
# Gravação de todas todas as tabelas em formato Delta 
bronze_delta_path = "/Volumes/main/default/faers_data/delta/bronze"

for table_name, df in bronze_dfs.items():
    output_name = f"{table_name.lower()}"
    delta_path = f"{bronze_delta_path}/{output_name}"

    (
        df
        .write
        .mode("overwrite")
        .format("delta")
        .option("overwriteSchema", "true")
        .save(delta_path)
    )

    print(f"Tabela Delta Bronze criada em: {delta_path}")

In [0]:
# validação final: schema correto, carregamento para formato delta e metadata
from pyspark.sql.functions import count, countDistinct

for table_name in selected_tables:
    delta_path = f"{bronze_delta_path}/{table_name.lower()}/"
    df = spark.read.format("delta").load(delta_path)

    print(f"\nMetadata Bronze: {table_name}")

    df.select(
        count("*").alias("total_rows"),
        count("source_file").alias("rows_with_source_file"),
        count("load_timestamp").alias("rows_with_load_timestamp"),
        countDistinct("source_file").alias("distinct_source_files")
    ).show(truncate=False)
    df.printSchema()
    display(
        df.groupBy("source_file")
          .count()
          .orderBy("source_file")
    )


## Conclusão da camada Bronze

Foram criadas quatro tabelas Bronze em formato Delta para as entidades `DEMO`, `DRUG`, `REAC` e `OUTC`.

Foram acrescentadas duas colunas de auditoria de metadados a cada tabela: `source_file` e `load_timestamp`.

A escrita foi feita em modo `overwrite`, uma vez que o projeto trabalha com um período fechado de dados, de `2022Q4` a `2023Q4`, permitindo recriar a camada Bronze de forma reprodutível durante o desenvolvimento.

